# CTC Model Training Pipeline
This notebook implements the Connectionist Temporal Classification (CTC) pipeline. 
Unlike the sliding window approach, this trains the `CTC_CRNN` sequentially on entire audio recordings using PyTorch's native `CTCLoss`.

In [ ]:
import sys
import shutil
import os

assert sys.version_info >= (3, 10)
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    !git clone https://github.com/stachuapa123/ASR_project.git
    %cd ASR_project
    # !git checkout <YOUR_BRANCH_NAME>  # Uncomment and set this to your branch if needed
    !pip install -q torchmetrics
    from google.colab import drive

    drive.mount("/content/drive")

    # Extract data securely if on Colab
    !mkdir -p /content/dane_audio_lokalnie
    !unzip -q "/content/drive/MyDrive/asr_data/501-1000.zip" -d "/content/dane_audio_lokalnie"
    DATA_DIR = "/content/dane_audio_lokalnie"
else:
    # Local path
    %load_ext autoreload
    %autoreload 2
    DATA_DIR = "../data/1-500"  # Update to your local subset or AutorskieDane

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from pathlib import Path

# Local CTC Imports
from src.constants import Constants as C
from src.ctc_parsers import CTCDataset, ctc_collate_fn
from src.augment import SpecAugment
from src.CTCModel import CTC_CRNN
from src.ctc_trainers import train_ctc_model

In [ ]:
# 1. Initialize Dynamic CTC Dataset
print(f"Loading dataset from: {DATA_DIR}")

# Enabling cache_mode=True to pre-compute the log-mel spectrograms into RAM.
# apply_augmentations=True means it will store both the base audio and 1 augmented copy per file in RAM,
# doubling the dataset size to severely increase epoch speeds on the free T4 Colab CPU.
dataset = CTCDataset(
    data_dir=DATA_DIR,
    cache_mode=True,
    apply_augmentations=True,
    max_files=None,  # Set to a small number for testing
    noiseprob=0.5,
    gainprob=0.5,
    tempo_prob=0.2,
    noise_level=(10, 30),
    gain_range=(-5, 5),
    tempo_range=(0.95, 1.05),
)

n_total = len(dataset)
print(f"Total authentic recordings (incl. augmentations) loaded to RAM: {n_total}")

In [ ]:
# 2. Train / Validation Split
torch.manual_seed(42)

n_val = max(1, int(0.15 * n_total))  # 15% validation
n_train = n_total - n_val

train_set, val_set = random_split(
    dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42)
)

print(f"Train files: {len(train_set)} | Val files: {len(val_set)}")

# 3. Collate via CTC Padded Loaders
# Note: num_workers>0 accelerates dynamic augmentation highly, but may error on Windows locally.
BATCH_SIZE = 8
NUM_WORKERS = 2 if IS_COLAB else 0

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=ctc_collate_fn,
    num_workers=NUM_WORKERS,
)
val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=ctc_collate_fn,
    num_workers=NUM_WORKERS,
)

In [ ]:
# 4. Model and Augmentation Initialization
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing on: {device}")

model = CTC_CRNN().to(device)

# Frequency/Time mask augmentation on the GPU sequence inside the training loop
mel_augmenter = SpecAugment(freq_mask_percent=0.2, time_mask_percent=0.125, p=0.5)

In [ ]:
# 5. Start Training!
EPOCHS = 100
CHECKPOINT_PATH = "trained_models/CTC_Baseline.pth"

model = train_ctc_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=EPOCHS,
    device=device,
    spec_augment=mel_augmenter,
    save_best_to=CHECKPOINT_PATH,
)

In [ ]:
# 6. Sanity check: Run an inference pass over one batch
mels, targets, in_lens, t_lens = next(iter(train_loader))
model.eval()

with torch.no_grad():
    mels = mels.to(device)
    logits = model(mels)  # CTC output format -> (Batch, Time/4, Classes + 1)

print(f"Input batch shape  (B, n_mels, T):   {mels.shape}")
print(f"Output logits shape (B, T/4, N_Cls): {logits.shape}")

# Decode first sample:
pred_indices = logits[0].argmax(dim=-1).cpu().numpy()
filtered_pred = [C.IDX2LABEL[idx] for idx in pred_indices if idx != C.N_CLASSES]

print("\nRaw predicted frames (blank is dropped):")
print(filtered_pred[:50])  # Truncated representation